# Publication notes
See [the project README](../README.md) and [setup instructions](../docs/setup.md). Outputs were cleared for publication; this file has not been rerun during packaging. Notebook 6 is the current integration development reference; supplied-run evidence and limitations are documented in the README.


# Notebook 5 — Information-aware probing with full history derivatives

**Updated formulation. Publication outputs are cleared; the uploaded source contained local execution outputs.**
Run this notebook in the same `quasistatic-contact-SLS` kernel as Notebook 1.
Save it alongside Notebooks 1–4 in
`~/projects/adaptive_contact_linear_pusher/notebooks/`.

The experiment plans six **position commands** for the original mass-only Drake pusher.
It compares manual probing, Notebook 5's frozen-generator/gain method, and the updated
method that differentiates the final uncertainty score through the entire forecast.
All methods use the same prior, physical limits, noise assumptions, and independent
position-based estimator. No true plant mass enters the planning functions.

**Sources and adaptations**

* Wei-Chen's `QuasidynamicLinearPusher`, `TrajOptProblem`, and `IterativeTO` provide the
  simulator and optimization framework. Costs and bounds use the analytical active-contact
  expressions derived from that model.
* Adaptive paper Eq. (13) supplies the posterior-generator recursion. Its regularized
  scalar implementation and radius gate are checked against the local adaptive repository.
  The terminal trace objective here is not the adaptive paper's example Eq. (41).
* *Safety at the Limits*, Sec. IV-B, Eqs. (26)–(27), motivates carrying derivatives of
  later uncertainty-related quantities with respect to earlier decisions into each local
  optimization. We adapt this idea to a **learning cost**, including derivatives of the
  scalar gain. We do not implement that paper's ADMM splitting, Eqs. (28)–(30), or its
  minimum-time formulation.
* The earlier notebooks supply the analytical mass model, position-only estimator,
  manual excitation sequence, and paired evaluation design. Notebook 4's velocity and
  friction model is outside this experiment.

**Scope:** active contact throughout the mass prior, forward object pushing, known stiffness,
fixed sample time, and bounded position noise. Command reversals are allowed when contact
remains active. Commands are discrete; actuator interpolation, approach/retreat, friction,
online replanning, and a complete adaptive SLS policy are not modeled.

The information forecast uses nominal sensitivity and is a planning surrogate. The
independent estimator reports actual mass intervals after execution. Numerical-model
allowances are empirical assumptions, not global proofs. A favorable predicted trace,
solver status, or test coverage must not be interpreted as a physical safety guarantee.

Use **Restart Kernel and Run All** for a clean run. Diagnostic cells contain checks for
you to execute locally. Flags in Cell 2 control batch evaluation, optional target mode,
and result export. No repository files are modified.


## 1. Imports and original repository interfaces


In [ ]:
import os
import ast
import importlib.util
import json
import sys
from pathlib import Path
from datetime import datetime, timezone

import numpy as np
import matplotlib.pyplot as plt
from scipy.optimize import linprog
import csv
import hashlib
import time

PROJECT = Path(os.environ.get("PUSHER_PROJECT", Path.cwd().parent)).expanduser().resolve()
CONTACT_REPO = Path(os.environ.get("CONTACT_SLS_REPO", Path.home() / "projects/quasistatic-contact-SLS")).expanduser().resolve()
ADAPTIVE_REPO = Path(os.environ.get("ADAPTIVE_SLS_REPO", Path.home() / "projects/adaptive_sls_complete")).expanduser().resolve()
ALGORITHM_DIR = CONTACT_REPO / "examples/planning_through_contact/algorithms"

try:
    from pydrake.all import QuasidynamicLinearPusher
except ImportError as exc:
    raise RuntimeError("Select the same working Drake kernel used for Notebook 1.") from exc

def load_source_module(name, path):
    if not path.is_file():
        raise FileNotFoundError(f"Expected source file: {path}")
    spec = importlib.util.spec_from_file_location(name, path)
    module = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(module)
    return module

# Load only these two modules, avoiding the package __init__ and unrelated controllers.
TrajOptProblem = load_source_module(
    "probe5_traj_opt_problem", ALGORITHM_DIR / "traj_opt_problem.py"
).TrajOptProblem
IterativeTO = load_source_module(
    "probe5_iterative_to", ALGORITHM_DIR / "iterative_to.py"
).IterativeTO

print("Python:", sys.executable)
print("Drake planning source:", ALGORITHM_DIR)
print("Adaptive source:", ADAPTIVE_REPO / "src/gpu_sls/gpu_sls.py")
print("Backend: original Drake IterativeTO. Optimization and execution happen when you run the cells.")


## 2. Prior, physical limits, and optimization settings

Let $\alpha=K_a h^2/(m+K_a h^2)=\alpha_c+r_\alpha\theta$, with initial
$\theta\in[-1,1]$. The nominal mass is obtained from the midpoint in alpha-space;
it is not the midpoint in mass. Measurements obey $y_k=o_k+n_k$, $|n_k|\le\epsilon_y$.
For an active transition, a uniform residual bound is
$b=(2-\alpha_-)\epsilon_y+\delta_o$. Neighboring residuals share sensor noise;
the bounded-data estimator does not assume statistical independence.

The new solver scales the full objective by $1/(1+\lambda)$ and commands by
`COMMAND_SCALE` for conditioning. This positive objective scaling does not change
the minimizers. Trust radii below are in meters. Proximal regularization is applied
in scaled-command coordinates.


In [ ]:
K_A, H, W = 1.0, 0.02, 0.1
N = 6
MASS_PRIOR = np.array([0.005, 0.030])  # kg
POSITION_NOISE = 5e-6                  # absolute position error, m
OBJECT_ALLOWANCE = 5e-6                # empirical one-step object discrepancy, m
PUSHER_ALLOWANCE = 120e-6              # empirical one-step pusher discrepancy, m
MIN_EXCITATION = 1e-3                  # ensures active contact throughout the prior
MAX_TRAVEL = 8e-3                      # relative to the actual initial object position
MAX_FORCE = 0.025                      # N, including numerical pusher allowance
MAX_COMMAND_STEP = 12e-3               # m per sample
MAX_COMMAND_OFFSET = 35e-3             # m from measured initial contact boundary

# Normalization and weights, not claimed to be physical energy units.
MOVE_SCALE, FORCE_SCALE, COMMAND_SCALE = 2e-3, 0.020, 10e-3
MOVE_WEIGHT, FORCE_WEIGHT, COMMAND_WEIGHT = 1.0, 0.25, 0.05
INFO_WEIGHTS = [0.0, 3.0, 30.0, 300.0, 3000.0]
OUTER_ITERS = 15
FEAS_TOL = 2e-8                        # m for the normalized constraint residuals
GAIN_REG = 1e-7                        # matches regularization in normalized coordinates

def alpha_from_mass(mass):
    return K_A * H**2 / (np.asarray(mass) + K_A * H**2)

def mass_from_alpha(alpha):
    return K_A * H**2 * (1.0 / np.asarray(alpha) - 1.0)

ALPHA_LO, ALPHA_HI = alpha_from_mass(MASS_PRIOR[::-1])
ALPHA_CENTER = (ALPHA_LO + ALPHA_HI) / 2
ALPHA_RADIUS = (ALPHA_HI - ALPHA_LO) / 2
NOMINAL_MASS = float(mass_from_alpha(ALPHA_CENTER))
RESIDUAL_BOUND = (2.0 - ALPHA_LO) * POSITION_NOISE + OBJECT_ALLOWANCE

print("Mass prior [g]:", MASS_PRIOR * 1e3)
print("Planning midpoint is in alpha-space; nominal mass [g]:", NOMINAL_MASS * 1e3)
print("Duration [s]:", N * H)
print("Uniform position-residual bound [um]:", RESIDUAL_BOUND * 1e6)
print("Physical velocity/friction are not included in this model.")


# New method settings; original frozen method remains at its earlier iteration budget.
FULL_MAX_ITERS = 80
TRUST_INITIAL = 2e-3
TRUST_MIN = 1e-8
TRUST_MAX = 12e-3
TRUST_SHRINK = 0.5
TRUST_EXPAND = 1.5
ACCEPT_RATIO = 0.10
EXPAND_RATIO = 0.75
PROX_REG = 0.10
PRED_REDUCTION_TOL = 1e-12
ACTUAL_REDUCTION_TOL = 1e-12
STATIONARITY_TOL = 1e-7  # scaled first-order linearized-feasible-set gap
TARGET_TOL = 1e-9       # dimensionless trace target tolerance
FD_COMMAND_STEP = 1e-7 # meters; diagnostic only
SENSITIVITY_THRESHOLD = 1e-10

RUN_DIAGNOSTICS = True
RUN_FROZEN_COMPARISON = True
RUN_BATCH = True
RUN_TARGET_MODE = False
TRACE_TARGET = 1e-3  # normalized trace, NOT mass interval width
SAVE_RESULTS = False

EVALUATION_MASSES = [0.005, 0.010, 0.015, 0.020, 0.030]
EVALUATION_SEEDS = [0, 1, 2, 3, 4]
print("New method: complete uncertainty gradient, trust region, full-objective acceptance.")
print("No experiments are precomputed in this notebook.")


## 3. Physical predictions, conservative envelopes, and disturbance cost

On active contact,
$\bar o_{k+1}=(1-\alpha_c)\bar o_k+\alpha_c(u_k+w)$.
Lower and upper object envelopes propagate the endpoints of the alpha prior and
$\pm\delta_o$. Contact is enforced against the upper envelope. Force is bounded by
$F_k^{ub}=K_a(1-\alpha_-)(u_k+w-L_k)+K_a\delta_p$.

The unchanged disturbance objective is
$$J_D=\frac1N\sum_k\left[
w_o\left(\frac{R_{k+1}-L_0}{s_o}\right)^2+
w_F\left(\frac{F_k^{ub}}{s_F}\right)^2+
w_u\left(\frac{u_k-u_{k-1}}{s_u}\right)^2\right].$$
The weights define preferences; this expression is not mechanical energy. The envelopes
cover the prior only conditional on the stated discrepancy and initial-position bounds.
The position, force, contact, and command constraints below are affine in the command vector.


In [ ]:
def analytical_step(state, command, mass):
    p, o = np.asarray(state, dtype=float)
    a = max(float(command) + W - o, 0.0)
    onext = o + float(alpha_from_mass(mass)) * a
    pnext = min(float(command), onext - W)
    return np.array([pnext, onext])

def planning_quantities(commands, y0):
    """Accept numeric or symbolic commands. All recursions are affine in commands."""
    lo, hi, nominal = [y0 - POSITION_NOISE], [y0 + POSITION_NOISE], [y0]
    excitation, f_upper, margins = [], [], []
    previous_command = y0 - W - POSITION_NOISE
    for u in commands:
        a = u + W - nominal[-1]
        excitation.append(a)
        gap_worst = u + W - hi[-1]
        fmax = K_A * (1 - ALPHA_LO) * (u + W - lo[-1]) + K_A * PUSHER_ALLOWANCE
        f_upper.append(fmax)
        lo.append((1 - ALPHA_LO) * lo[-1] + ALPHA_LO * (u + W) - OBJECT_ALLOWANCE)
        hi.append((1 - ALPHA_HI) * hi[-1] + ALPHA_HI * (u + W) + OBJECT_ALLOWANCE)
        nominal.append((1 - ALPHA_CENTER) * nominal[-1] + ALPHA_CENTER * (u + W))
        # All margins expressed in meters for a common feasibility tolerance.
        margins += [
            gap_worst - MIN_EXCITATION,
            MAX_TRAVEL - (hi[-1] - (y0 - POSITION_NOISE)),
            (MAX_FORCE - fmax) / K_A,
            MAX_COMMAND_STEP - (u - previous_command),
            MAX_COMMAND_STEP + (u - previous_command),
            MAX_COMMAND_OFFSET - (u - (y0 - W)),
            u - (y0 - W),
        ]
        previous_command = u
    return dict(lo=lo, hi=hi, nominal=nominal, a=excitation, force=f_upper, margins=margins)

def feasible(commands, y0):
    return np.min(np.asarray(planning_quantities(commands, y0)["margins"], float)) >= -FEAS_TOL

def disturbance_cost(commands, y0):
    q = planning_quantities(commands, y0)
    previous = y0 - W - POSITION_NOISE
    cost = 0.0
    for k, u in enumerate(commands):
        cost += MOVE_WEIGHT * ((q["hi"][k+1] - (y0-POSITION_NOISE)) / MOVE_SCALE)**2
        cost += FORCE_WEIGHT * (q["force"][k] / FORCE_SCALE)**2
        cost += COMMAND_WEIGHT * ((u - previous) / COMMAND_SCALE)**2
        previous = u
    return cost / N

print("Physical model, envelope, constraints, and disturbance cost defined.")


## 4. Original Drake plant (function definitions only)


In [ ]:
class DrakePlant:
    def __init__(self, mass):
        self.system = QuasidynamicLinearPusher(
            k_a=K_A, m_o=float(mass), w=W, dt=H, dynamics_smoothing=0.0
        )
        self.context = self.system.CreateDefaultContext()

    def step(self, state, command):
        self.context.SetDiscreteState(np.asarray(state, dtype=float))
        self.system.get_input_port().FixValue(self.context, np.array([float(command)]))
        update = self.system.EvalUniquePeriodicDiscreteUpdate(self.context)
        self.context.SetDiscreteState(update)
        return self.context.get_discrete_state_vector().CopyToVector()


print("DrakePlant defined. Calibration runs in Cell 8 if enabled.")


## 5. Explicit generator reference and adaptive source check interface

For observable scalar sensitivity $H_k=r_\alpha a_k$, set $W_k=b/H_k$ and
$K_k=q_k/(q_k+W_k^2+\varepsilon_K)$, where $q_k=G_kG_k^\top$.
Adaptive Eq. (13) becomes $G_{k+1}=[(1-K_k)G_k,\;K_kW_k]$.
Retain the two radius-contraction checks from the original adapter. A rejected update
uses $K_k=0$. This introduces switching surfaces: derivatives are branch-local.

The reference below reads only `adaptive_posterior_factors` from the local source and
evaluates it with NumPy. It does not import JAX or the full adaptive controller.


In [ ]:
def posterior_update(generators, sensitivity, residual_bound=RESIDUAL_BOUND):
    G = np.asarray(generators, float)
    q = float(G @ G)
    radius = float(np.abs(G).sum())
    if abs(sensitivity) < 1e-10:
        return np.r_[G, 0.0], 0.0
    noise_map = residual_bound / sensitivity
    L = q / (q + noise_map**2 + GAIN_REG)
    Gplus = np.r_[(1-L)*G, L*noise_map]
    if not (abs(1-L)*radius < radius and np.abs(Gplus).sum() < radius):
        return np.r_[G, 0.0], 0.0
    return Gplus, L / sensitivity

# Read only the pure numerical function from the verified local repo.
# No JAX import, repository modification, or full adaptive controller import is needed.
source_path = ADAPTIVE_REPO / "src/gpu_sls/gpu_sls.py"
tree = ast.parse(source_path.read_text())
source_function = next(
    node for node in tree.body
    if isinstance(node, ast.FunctionDef) and node.name == "adaptive_posterior_factors"
)
source_function.decorator_list = []
namespace = {"jnp": np}
exec(compile(ast.Module(body=[source_function], type_ignores=[]), str(source_path), "exec"), namespace)
reference_update = namespace["adaptive_posterior_factors"]



def generator_forecast(commands, y0):
    excitation = np.asarray(planning_quantities(commands, y0)["a"], float)
    G = np.array([1.0])
    histories, raw_gains, traces, radii = [], [], [1.0], [1.0]
    for a in excitation:
        histories.append(G.copy())
        G, gain = posterior_update(G, ALPHA_RADIUS * max(a, 0.0))
        raw_gains.append(gain)
        traces.append(float(G @ G))
        radii.append(float(np.abs(G).sum()))
    return dict(G=histories, gains=np.asarray(raw_gains), traces=np.asarray(traces),
                radii=np.asarray(radii))

print("Generator reference and local adaptive-source function loaded.")


## 6. New: compact uncertainty recursion and derivatives through all earlier commands

Without generator reduction, the scalar summaries exactly obey
$$q_{k+1}=(1-K_k)^2q_k+K_k^2W_k^2,\qquad
r_{k+1}=|1-K_k|r_k+|K_kW_k|.$$
Here $q_k$ is the trace score; $r_k=\|G_k\|_1$ controls the interval-radius gate.
They are not a covariance and standard deviation.

For $S_{k,j}=\partial\bar o_k/\partial u_j$ and $D_{k,j}=\partial H_k/\partial u_j$,
$$S_{k+1,j}=(1-\alpha_c)S_{k,j}+\alpha_c\mathbf1_{k=j},\qquad
D_{k,j}=r_\alpha(\mathbf1_{k=j}-S_{k,j}).$$
If $A_k^q=\partial q_{k+1}/\partial q_k$ and $B_k^q=\partial q_{k+1}/\partial H_k$,
$$T_{k+1,j}=A_k^q T_{k,j}+B_k^qD_{k,j},\qquad
T_{0,j}=0,\qquad \nabla_Uq_N=T_{N,:}.$$
The code differentiates through the gain as well as through $q_k$ and the physical
position prediction. Within a rejected branch, $A_k^q=1$, $B_k^q=0$. At a switch,
an ordinary smooth derivative need not exist; candidate forecasts are always recomputed.

Explicit accepted-branch derivatives use $Z=q+W^2+\varepsilon_K$ and
$C=KW^2-(1-K)q$:
$$A^q=(1-K)^2+2C\frac{W^2+\varepsilon_K}{Z^2},\qquad
B^q=2C\frac{2qW^2}{HZ^2}-\frac{2K^2W^2}{H}.$$
These are our scalar full-gain adaptation of the cross-time derivative principle;
they are not formulas copied from the coworker's paper.


In [ ]:
def compact_posterior(q, radius, sensitivity):
    if sensitivity <= SENSITIVITY_THRESHOLD:
        return dict(q=q, radius=radius, K=0.0, raw_gain=0.0, Aq=1.0, Bq=0.0,
                    active=False, gate_margin=np.nan)
    noise_map = RESIDUAL_BOUND / sensitivity
    denom = q + noise_map**2 + GAIN_REG
    gain = q / denom
    pre_radius = abs(1.0-gain) * radius
    new_radius = pre_radius + abs(gain*noise_map)
    gate_margin = radius - new_radius
    active = bool(pre_radius < radius and new_radius < radius)
    if not active:
        return dict(q=q, radius=radius, K=0.0, raw_gain=0.0, Aq=1.0, Bq=0.0,
                    active=False, gate_margin=gate_margin)
    new_q = (1-gain)**2*q + (gain*noise_map)**2
    correction = gain*noise_map**2 - (1-gain)*q
    gain_q = (noise_map**2 + GAIN_REG) / denom**2
    gain_H = 2*q*noise_map**2 / (sensitivity*denom**2)
    Aq = (1-gain)**2 + 2*correction*gain_q
    Bq = 2*correction*gain_H - 2*gain**2*noise_map**2/sensitivity
    return dict(q=new_q, radius=new_radius, K=gain, raw_gain=gain/sensitivity,
                Aq=Aq, Bq=Bq, active=True, gate_margin=gate_margin)

def forecast_with_gradient(commands, y0):
    commands = np.asarray(commands, float)
    if commands.shape != (N,) or not np.isfinite(commands).all():
        raise ValueError(f"Expected {N} finite commands.")
    nominal = float(y0)
    S, T = np.zeros(N), np.zeros(N)
    q, radius = 1.0, 1.0
    traces, radii, positions = [q], [radius], [nominal]
    trace_grads = [T.copy()]
    sensitivities, gains, gates, margins, As, Bs = [], [], [], [], [], []
    for k, u in enumerate(commands):
        direction = np.eye(N)[k]
        a = float(u + W - nominal)
        if a <= 0:
            raise ValueError("This forecast requires active contact; candidate is outside its domain.")
        sensitivity = ALPHA_RADIUS*a
        D = ALPHA_RADIUS*(direction-S)
        post = compact_posterior(q, radius, sensitivity)
        T = post["Aq"]*T + post["Bq"]*D
        q, radius = float(post["q"]), float(post["radius"])
        nominal = (1-ALPHA_CENTER)*nominal + ALPHA_CENTER*(u+W)
        S = (1-ALPHA_CENTER)*S + ALPHA_CENTER*direction
        traces.append(q); radii.append(radius); positions.append(nominal)
        trace_grads.append(T.copy()); sensitivities.append(sensitivity)
        gains.append(post["K"]); gates.append(post["active"])
        margins.append(post["gate_margin"]); As.append(post["Aq"]); Bs.append(post["Bq"])
    return dict(traces=np.asarray(traces), radii=np.asarray(radii),
                positions=np.asarray(positions), trace_gradients=np.asarray(trace_grads),
                gradient=T.copy(), sensitivities=np.asarray(sensitivities),
                gains=np.asarray(gains), gates=np.asarray(gates, bool),
                gate_margins=np.asarray(margins), Aq=np.asarray(As), Bq=np.asarray(Bs))

print("Full-history forecast and branch-local analytical gradient defined.")


## 7. Full objective, exact disturbance derivatives, and manual warm start

Use $J=J_D+\lambda q_N$, equivalent to $J_D-\lambda(1-q_N)$ up to a constant.
The physical envelope and stage residuals are affine, so $J_D=\|v(U)\|_2^2$ has
gradient $2J_v^\top v$ and Hessian $2J_v^\top J_v$.
The affine maps below are obtained by evaluating their basis coefficients; these
are exact affine coefficient constructions, not finite-difference information gradients.

The planner knows $y_0$ and the mass prior. It does not use the evaluation masses.
Manual commands reconstruct the six earlier excitation sizes around the nominal
trajectory in open loop; no claim is made that the previous notebook's execution policy
was identical.


In [ ]:
def cost_residuals(commands, y0):
    quantities = planning_quantities(commands, y0)
    previous = y0-W-POSITION_NOISE
    residuals = []
    for k, u in enumerate(commands):
        residuals.extend([
            np.sqrt(MOVE_WEIGHT/N)*(quantities["hi"][k+1]-(y0-POSITION_NOISE))/MOVE_SCALE,
            np.sqrt(FORCE_WEIGHT/N)*quantities["force"][k]/FORCE_SCALE,
            np.sqrt(COMMAND_WEIGHT/N)*(u-previous)/COMMAND_SCALE,
        ])
        previous = u
    return np.asarray(residuals, float)

def affine_jacobian(function, y0):
    anchor = np.full(N, y0-W)
    value = np.asarray(function(anchor, y0), float)
    columns = []
    for direction in np.eye(N):
        columns.append((np.asarray(function(anchor+COMMAND_SCALE*direction, y0), float)-value)/COMMAND_SCALE)
    return np.column_stack(columns)

def margin_vector(commands, y0):
    return np.asarray(planning_quantities(commands, y0)["margins"], float)

Y0 = 0.20
J_RESIDUAL = affine_jacobian(cost_residuals, Y0)
J_MARGIN = affine_jacobian(margin_vector, Y0)

def disturbance_gradient(commands, y0):
    return 2*J_RESIDUAL.T @ cost_residuals(commands, y0)

def full_objective(commands, y0, weight):
    score = forecast_with_gradient(commands, y0)["traces"][-1]
    return float(disturbance_cost(commands, y0) + weight*score)

def full_objective_gradient(commands, y0, weight, forecast=None):
    if forecast is None:
        forecast = forecast_with_gradient(commands, y0)
    return disturbance_gradient(commands, y0) + weight*forecast["gradient"]

def manual_commands(y0):
    o, commands = float(y0), []
    for a in np.array([5, 10, 15, 20, 10, 20])*1e-3:
        commands.append(o-W+a)
        o += ALPHA_CENTER*a
    return np.asarray(commands)

MANUAL = manual_commands(Y0)
assert feasible(MANUAL, Y0), "The manual baseline is infeasible under the selected budgets."
print("Manual upper travel [mm]:", 1e3*(planning_quantities(MANUAL,Y0)["hi"][-1]-(Y0-POSITION_NOISE)))
print("Manual predicted trace:", forecast_with_gradient(MANUAL,Y0)["traces"][-1])
print("Manual predicted radius:", forecast_with_gradient(MANUAL,Y0)["radii"][-1])


## 8. Diagnostics for you to run: source parity, gradient, and Drake calibration

These checks are included for local validation; they were not run when this notebook
was generated. A finite-difference comparison is interpreted as a smooth derivative
check only when both perturbations retain the reference gate pattern. A matching
pattern at the endpoints is a useful diagnostic, not a proof of smoothness everywhere.
No passing outcome is assumed in the later reporting.


In [ ]:
def gradient_diagnostic(commands, y0, step=FD_COMMAND_STEP):
    forecast = forecast_with_gradient(commands, y0)
    finite_difference = np.empty(N)
    same_branch = np.empty(N, bool)
    for j in range(N):
        delta = np.eye(N)[j]*step
        plus = forecast_with_gradient(commands+delta, y0)
        minus = forecast_with_gradient(commands-delta, y0)
        finite_difference[j] = (plus["traces"][-1]-minus["traces"][-1])/(2*step)
        same_branch[j] = (np.array_equal(plus["gates"], forecast["gates"]) and
                          np.array_equal(minus["gates"], forecast["gates"]))
    return dict(analytic=forecast["gradient"], finite_difference=finite_difference,
                same_branch=same_branch,
                scaled_error=COMMAND_SCALE*np.abs(forecast["gradient"]-finite_difference))

DIAGNOSTICS = {}
if RUN_DIAGNOSTICS:
    for label, sensitivity, noise in [("no sensitivity",0.,.1), ("informative",1.,.1), ("too noisy",1.,2.)]:
        observable = abs(sensitivity) >= 1e-10
        Pi = np.array([[float(observable)]])
        Wobs = np.array([[noise/sensitivity if observable else 0.]])
        _, _, expected, _ = reference_update(np.array([[1.]]), Pi, Wobs, GAIN_REG, True)
        actual, _ = posterior_update(np.array([1.]), sensitivity, noise)
        np.testing.assert_allclose(actual, expected.ravel(), atol=1e-12)
        print(label, "source parity passed")
    explicit = generator_forecast(MANUAL, Y0)
    compact = forecast_with_gradient(MANUAL, Y0)
    for key in ["traces", "radii"]:
        np.testing.assert_allclose(explicit[key], compact[key], rtol=1e-11, atol=1e-13)
    DIAGNOSTICS["gradient"] = gradient_diagnostic(MANUAL, Y0)
    diagnostic = DIAGNOSTICS["gradient"]
    print("Coordinate / analytical dqN/du / central difference / unchanged gates")
    for j in range(N):
        print(j, diagnostic["analytic"][j], diagnostic["finite_difference"][j], diagnostic["same_branch"][j])
    smooth = diagnostic["same_branch"]
    if smooth.any():
        np.testing.assert_allclose(COMMAND_SCALE*diagnostic["analytic"][smooth],
                                   COMMAND_SCALE*diagnostic["finite_difference"][smooth],
                                   rtol=5e-4, atol=1e-7)
    print("Coordinates excluded due to gate changes:", np.flatnonzero(~smooth))
    max_calibration = np.zeros(2)
    for mass in [MASS_PRIOR[0], .015, MASS_PRIOR[1]]:
        plant = DrakePlant(mass)
        for o in [.15, .20, .30]:
            for a in [.001, .005, .010, .025]:
                state, u = np.array([o-W,o]), o-W+a
                difference = np.abs(plant.step(state,u)-analytical_step(state,u,mass))
                max_calibration = np.maximum(max_calibration,difference)
    DIAGNOSTICS["calibration"] = max_calibration
    print("Calibration max [pusher, object] discrepancy [um]:", max_calibration*1e6)
    assert np.all(max_calibration <= np.array([PUSHER_ALLOWANCE,OBJECT_ALLOWANCE]))
    print("Enabled checks completed. Calibration is not a global error proof.")
else:
    print("Diagnostics skipped by RUN_DIAGNOSTICS=False; no validation claim is made.")


## 9. Drake quadratic-subproblem interface and original-method baseline

All symbolic costs and constraints are functions of the complete command vector.
The original Drake model supplies nominal state dynamics inside `IterativeTO`.
Both `use_true_dynamics_*` flags are **False** because this model is already created
with zero dynamics smoothing and does not provide geometry-smoothing accessors.

The frozen baseline retains its original information approximation, manual warm start,
iteration budget, and full-objective line search. Its stopping reason is recorded, not
converted into a convergence claim. The new solver uses explicit linear command trust
bounds, avoiding the incompatible smoothing path.


In [ ]:
def make_drake_problem(y0):
    model = QuasidynamicLinearPusher(k_a=K_A, m_o=NOMINAL_MASS, w=W, dt=H,
                                   dynamics_smoothing=0.0)
    problem = TrajOptProblem(system=model, num_steps=N)
    problem.SetInitialState([y0-W-2*POSITION_NOISE, y0])
    symbols = [problem.input(k)[0] for k in range(N)]
    for margin in planning_quantities(symbols,y0)["margins"]:
        problem.AddConstraint(margin, lb=0.0, ub=np.inf)
    return problem, symbols

def solve_drake_problem(problem, reference, legacy_trust=False):
    trajectory = IterativeTO(
        problem, us_init=np.asarray(reference)[:,None],
        use_contact_trust_region=False,
        use_true_dynamics_for_rollout=False,
        use_true_dynamics_for_constraint=False,
        trust_region_covar=np.eye(3)/(0.020**2) if legacy_trust else None,
        max_iters=8, tol=1e-9,
    )
    candidate = np.asarray(trajectory.us,float).ravel()
    if candidate.shape != (N,) or not np.isfinite(candidate).all():
        raise RuntimeError("Drake returned invalid command values.")
    return candidate

def frozen_cost(commands, y0, weight, frozen):
    a = planning_quantities(commands,y0)["a"]
    cost = disturbance_cost(commands,y0)
    for k in range(N):
        q = float(frozen["G"][k] @ frozen["G"][k])
        ell = float(frozen["gains"][k])
        cost += weight*(q*(1-ell*ALPHA_RADIUS*a[k])**2+(ell*RESIDUAL_BOUND)**2-q)
    return cost/(1+weight)

def solve_frozen_drake(reference, y0, weight, frozen):
    problem, symbols = make_drake_problem(y0)
    problem.AddCost(frozen_cost(symbols,y0,weight,frozen))
    return solve_drake_problem(problem, reference, legacy_trust=True)

def optimize_frozen(y0, weight, initial):
    started = time.perf_counter()
    reference = np.asarray(initial,float).copy()
    history, logs = [full_objective(reference,y0,weight)], []
    status = "iteration_limit"
    for iteration in range(OUTER_ITERS):
        frozen = generator_forecast(reference,y0)
        candidate = solve_frozen_drake(reference,y0,weight,frozen)
        accepted = False
        for scale in [1., .5, .25, .125, .0625, .03125]:
            trial = reference+scale*(candidate-reference)
            if feasible(trial,y0) and full_objective(trial,y0,weight) < history[-1]-1e-10:
                change = float(np.max(np.abs(trial-reference)))
                reference = trial
                history.append(full_objective(reference,y0,weight))
                accepted = True
                break
        logs.append(dict(iteration=iteration, accepted=accepted, objective=history[-1]))
        if not accepted:
            status = "no_accepted_step_not_a_stationarity_certificate"; break
        if change < 1e-7:
            status = "small_step_not_a_stationarity_certificate"; break
    return dict(commands=reference, objective_history=np.asarray(history), logs=logs,
                status=status, seconds=time.perf_counter()-started, final_gap=np.nan)

print("Drake subproblem interface and frozen baseline defined.")


## 10. New: full-gradient trust-region optimizer

At iterate $U^s$, recompute the whole forecast and $g^s=\nabla_Uq_N(U^s)$.
Write $s_\lambda=1+\lambda$. Solve
$$\min_{\Delta U}\ \frac{J_D(U^s+\Delta U)+\lambda(q_N^s+(g^s)^\top\Delta U)}{s_\lambda}
+\frac{\beta}{2}\left\|\frac{\Delta U}{s_u}\right\|_2^2,$$
subject to the physical constraints and $\|\Delta U\|_\infty\le\rho_s$.
Here $\beta>0$ is proximal regularization, not an information weight. Accept a step only
after evaluating the actual full objective and physical feasibility. Use the ratio of
actual to predicted decrease to adjust the trust radius.

For a local stationarity diagnostic, solve a small linear program minimizing the
full first-order objective over the physical feasible set. In optional target mode,
also include the target's local linearization. The resulting gap is meaningful for
the selected smooth branch and constraints; it is not a global-optimality certificate.
Radius exhaustion, QP failure, iteration limit, and stationary-branch status remain distinct.

An optional target $q_N\le q_{target}$ uses
$(g^s)^\top\Delta U\le q_{target}-q_N^s$, the cost-application counterpart of the
paper's cross-time constraint correction. It starts from a genuinely target-feasible plan
and checks the nonlinear target after every step. No feasibility-restoration claim is made.


In [ ]:
def local_model_value(commands, reference, y0, weight, current_forecast):
    delta = np.asarray(commands)-reference
    trace_linear = current_forecast["traces"][-1] + current_forecast["gradient"] @ delta
    return float((disturbance_cost(commands,y0)+weight*trace_linear)/(1+weight)
                 + .5*PROX_REG*np.sum((delta/COMMAND_SCALE)**2))

def solve_gradient_drake(reference, y0, weight, current_forecast, radius, target=None):
    problem, symbols = make_drake_problem(y0)
    delta = [symbols[k]-float(reference[k]) for k in range(N)]
    linear_trace = float(current_forecast["traces"][-1]) + sum(
        float(current_forecast["gradient"][k])*delta[k] for k in range(N))
    cost = (disturbance_cost(symbols,y0)+weight*linear_trace)/(1+weight)
    cost += .5*PROX_REG*sum((d/COMMAND_SCALE)**2 for d in delta)
    problem.AddCost(cost)
    for d in delta:
        problem.AddConstraint(d, lb=-float(radius), ub=float(radius))
    if target is not None:
        problem.AddConstraint(linear_trace, lb=-np.inf, ub=float(target))
    return solve_drake_problem(problem, reference)

def stationarity_gap(commands, y0, weight, forecast, target=None):
    # Variable z=(v-commands)/COMMAND_SCALE, with v in the affine feasible set.
    gradient = full_objective_gradient(commands,y0,weight,forecast)/(1+weight)
    A = -J_MARGIN*COMMAND_SCALE
    upper = margin_vector(commands,y0)
    if target is not None:
        A = np.vstack((A,forecast["gradient"]*COMMAND_SCALE))
        upper = np.r_[upper,target-forecast["traces"][-1]]
    lp = linprog(gradient*COMMAND_SCALE, A_ub=A, b_ub=upper,
                 bounds=[(None,None)]*N, method="highs")
    if not lp.success:
        return np.nan
    return max(0.0,float(-lp.fun))

def optimize_full_history(y0, weight, initial, target=None):
    started = time.perf_counter()
    reference = np.asarray(initial,float).copy()
    if not feasible(reference,y0):
        raise ValueError("A physically feasible initial command sequence is required.")
    current = forecast_with_gradient(reference,y0)
    if target is not None and current["traces"][-1] > target:
        raise ValueError("Target mode requires an initially target-feasible plan; no restoration phase is implemented.")
    radius = TRUST_INITIAL
    current_value = full_objective(reference,y0,weight)
    history, logs = [current_value], []
    status = "iteration_limit"
    final_gap = np.nan
    for iteration in range(FULL_MAX_ITERS):
        final_gap = stationarity_gap(reference,y0,weight,current,target)
        if np.isfinite(final_gap) and final_gap <= STATIONARITY_TOL:
            status = "small_first_order_gap_on_current_branch"; break
        base = current_value/(1+weight)
        try:
            candidate = solve_gradient_drake(reference,y0,weight,current,radius,target)
        except RuntimeError as exc:
            logs.append(dict(iteration=iteration, accepted=False, reason="inner_solver_error",
                             detail=str(exc), trust_mm=radius*1e3, objective=current_value,
                             stationarity_gap=final_gap))
            radius *= TRUST_SHRINK
            if radius < TRUST_MIN:
                status = "inner_solver_failure_at_minimum_radius"; break
            continue
        delta = candidate-reference
        step = float(np.max(np.abs(delta)))
        physical_ok = bool(feasible(candidate,y0) and step <= radius+FEAS_TOL)
        predicted = base-local_model_value(candidate,reference,y0,weight,current)
        trial = None
        actual, ratio = np.nan, np.nan
        target_ok = target is None
        if physical_ok:
            trial = forecast_with_gradient(candidate,y0)
            target_ok = target is None or trial["traces"][-1] <= target+TARGET_TOL
            candidate_value = float(disturbance_cost(candidate,y0)+weight*trial["traces"][-1])
            actual = (current_value-candidate_value)/(1+weight)
            if predicted > PRED_REDUCTION_TOL:
                ratio = actual/predicted
        accepted = bool(physical_ok and target_ok and predicted > PRED_REDUCTION_TOL
                        and actual > ACTUAL_REDUCTION_TOL and ratio >= ACCEPT_RATIO)
        gate_changes = int(np.count_nonzero(trial["gates"] != current["gates"])) if trial is not None else 0
        record = dict(iteration=iteration, accepted=accepted,
                      objective_before=current_value, trust_mm=radius*1e3, step_mm=step*1e3,
                      predicted_reduction=predicted, actual_reduction=actual, ratio=ratio,
                      physical_ok=physical_ok, target_ok=target_ok, gate_changes=gate_changes,
                      stationarity_gap=final_gap)
        if accepted:
            reference, current, current_value = candidate, trial, candidate_value
            history.append(current_value)
            if ratio >= EXPAND_RATIO and step >= .8*radius:
                radius = min(TRUST_MAX,radius*TRUST_EXPAND)
        else:
            radius *= TRUST_SHRINK
        record["objective_after"] = current_value
        logs.append(record)
        if radius < TRUST_MIN:
            status = "trust_radius_exhausted_not_converged"; break
    final_forecast = forecast_with_gradient(reference,y0)
    final_gap = stationarity_gap(reference,y0,weight,final_forecast,target)
    return dict(commands=reference, objective_history=np.asarray(history), logs=logs,
                status=status, seconds=time.perf_counter()-started, final_gap=final_gap,
                forecast=final_forecast, trace_target=target)

print("Full-history optimizer defined. No optimization has run in this cell.")


## 11. Run the paired planning comparison

This cell actually runs the optimizers on your machine. Each method starts from the same
manual trajectory for each weight. The methods have different iteration budgets; runtime
and status are reported, so a comparison is not a matched-compute experiment.
An exception is recorded and the failed strategy is excluded explicitly, never silently
relabelled as a successful optimized plan.


In [ ]:
PLANS = {"Manual": MANUAL.copy()}
PLAN_META = {"Manual": dict(method="manual", weight=None)}
OPT_RESULTS, PLAN_FAILURES = {}, []
for weight in INFO_WEIGHTS:
    methods = [("Full history",optimize_full_history)]
    if RUN_FROZEN_COMPARISON:
        methods.insert(0,("Frozen",optimize_frozen))
    for method, optimizer in methods:
        name = f"{method} lambda={weight:g}"
        print("Planning:",name,flush=True)
        try:
            result = optimizer(Y0,weight,MANUAL)
        except Exception as exc:
            PLAN_FAILURES.append(dict(strategy=name,error=f"{type(exc).__name__}: {exc}"))
            print("FAILED:",PLAN_FAILURES[-1]["error"],flush=True)
            continue
        commands = result["commands"]
        if not feasible(commands,Y0):
            raise RuntimeError(f"Returned plan is infeasible: {name}")
        PLANS[name], OPT_RESULTS[name] = commands, result
        PLAN_META[name] = dict(method=method,weight=float(weight))
        forecast = forecast_with_gradient(commands,Y0)
        quantities = planning_quantities(commands,Y0)
        print(f"  accepted={len(result['objective_history'])-1}, qN={forecast['traces'][-1]:.6g}, "
              f"radius={forecast['radii'][-1]:.6g}, upper travel="
              f"{1e3*(quantities['hi'][-1]-(Y0-POSITION_NOISE)):.4f} mm, time={result['seconds']:.2f}s")
        print("  status:",result["status"],"gap:",result["final_gap"],flush=True)
print("Planning exceptions:",len(PLAN_FAILURES))
print("Plans are fixed before evaluating hidden masses and noise seeds.")


## 12. Inspect objective histories, gate changes, and stopping reasons


In [ ]:
fig, ax = plt.subplots(figsize=(10,4))
for name, result in OPT_RESULTS.items():
    weight = PLAN_META[name]["weight"]
    history = result["objective_history"]/(1+weight)
    ax.plot(np.arange(len(history)),history,marker=".",label=name)
    failures = sum(log.get("reason")=="inner_solver_error" for log in result["logs"])
    switches = sum(log.get("gate_changes",0) for log in result["logs"] if log.get("accepted",False))
    print(name,"|",result["status"],"| gap",result["final_gap"],
          "| inner errors",failures,"| accepted gate changes",switches)
ax.set(xlabel="Accepted update",ylabel="Scaled objective J/(1+lambda)",
       title="Within each curve: improvement; different weights define different objectives")
ax.grid(alpha=.3)
if OPT_RESULTS: ax.legend(fontsize=7,ncol=2)
plt.tight_layout(); plt.show()
print("No accepted step / a small trust radius does not certify convergence.")
print("Use gradient_diagnostic(PLANS[name], Y0) for an additional local derivative check.")


## 13. Optional uncertainty-target formulation

With `RUN_TARGET_MODE=True`, minimize $J_D$ subject to $q_N(U)\le q_{target}$.
Use the linearized target within each subproblem and check the nonlinear target afterward.
Choose an initial feasible plan using **predicted** scores only. If none is available,
report that no starting plan was found; this is not proof that the target is impossible.
The target is a normalized trace score, not a certified mass-width requirement.


In [ ]:
TARGET_RESULT = None
if RUN_TARGET_MODE:
    eligible = [(name,u) for name,u in PLANS.items()
                if forecast_with_gradient(u,Y0)["traces"][-1] <= TRACE_TARGET]
    if not eligible:
        print("No target-feasible starting plan found. Adjust the declared target or generate another feasible plan.")
    else:
        start_name, start = min(eligible,key=lambda pair:disturbance_cost(pair[1],Y0))
        print("Target-mode initial plan:",start_name,"target:",TRACE_TARGET)
        TARGET_RESULT = optimize_full_history(Y0,0.,start,target=TRACE_TARGET)
        name = f"Trace target={TRACE_TARGET:g}"
        PLANS[name] = TARGET_RESULT["commands"]
        OPT_RESULTS[name] = TARGET_RESULT
        PLAN_META[name] = dict(method="target",weight=0.,trace_target=TRACE_TARGET)
        print("Status:",TARGET_RESULT["status"],"final score:",TARGET_RESULT["forecast"]["traces"][-1])
else:
    print("Optional target mode disabled. Weighted-cost comparison remains the main experiment.")


## 14. Independent position-only mass estimator

For $a_y=u_k+w-y_k>\epsilon_y$ and $d_y=y_{k+1}-y_k$, intersect the previous alpha interval with
$$\frac{d_y-2\epsilon_y-\delta_o}{a_y-\epsilon_y}\le\alpha\le
\frac{d_y+2\epsilon_y+\delta_o}{a_y+\epsilon_y}.$$
Convert decreasingly through $m=K_a h^2(1/\alpha-1)$ to obtain the mass interval.
Neither true mass, contact force, nor the planner's forecast is passed to this estimator.
An empty interval is reported as failure; the prior is not silently reset.


In [ ]:
def identify_positions(commands, measurements):
    low, high = float(ALPHA_LO), float(ALPHA_HI)
    intervals = [[low, high]]
    accepted = 0
    for k, u in enumerate(commands):
        a_y = u + W - measurements[k]
        d_y = measurements[k+1] - measurements[k]
        if a_y > POSITION_NOISE + 1e-12:
            low = max(low, (d_y-2*POSITION_NOISE-OBJECT_ALLOWANCE)/(a_y-POSITION_NOISE))
            high = min(high, (d_y+2*POSITION_NOISE+OBJECT_ALLOWANCE)/(a_y+POSITION_NOISE))
            accepted += 1
        if low > high + 1e-12:
            raise RuntimeError(f"Empty parameter interval at transition {k}; check model/noise assumptions.")
        intervals.append([low, high])
    intervals = np.asarray(intervals)
    mass_intervals = np.column_stack((mass_from_alpha(intervals[:,1]), mass_from_alpha(intervals[:,0])))
    return dict(alpha_intervals=intervals, mass_intervals=mass_intervals,
                mass_estimate=float(mass_intervals[-1].mean()), accepted=accepted)



if RUN_DIAGNOSTICS:
    from itertools import product
    checked = 0
    for mass in [.005,.015,.030]:
        alpha = float(alpha_from_mass(mass))
        for a in [.001,.005,.020]:
            for n0,n1,delta in product([-POSITION_NOISE,POSITION_NOISE],
                                      [-POSITION_NOISE,POSITION_NOISE],
                                      [-OBJECT_ALLOWANCE,OBJECT_ALLOWANCE]):
                o0=.2
                fit=identify_positions([o0-W+a],[o0+n0,o0+alpha*a+delta+n1])
                lo,hi=fit["alpha_intervals"][-1]
                assert lo-1e-12 <= alpha <= hi+1e-12
                checked += 1
    print("Endpoint-noise/model-error corner checks passed:",checked)
else:
    print("Estimator defined; corner diagnostics disabled.")


## 15. Execute the plans in Drake and identify mass

The hidden mass is used **only by the execution/evaluation function**. Each paired method
receives the same initial measurement and sensor noise. Initial true position is selected
consistently with that measurement. Contact forces are model-derived evaluation quantities;
they are not measurements supplied to the estimator. Report error as well as interval width.


In [ ]:
def execute_and_evaluate(commands, mass, seed, plant_factory=DrakePlant):
    rng = np.random.default_rng(seed)
    initial_noise = rng.uniform(-POSITION_NOISE, POSITION_NOISE)
    o0 = Y0 - initial_noise          # same observed initial Y0 for every compared strategy
    x = np.array([Y0-W-2*POSITION_NOISE, o0])
    plant = plant_factory(mass)
    states, forces, discrepancies = [x.copy()], [], []
    for u in commands:
        expected = analytical_step(x, u, mass)
        next_x = plant.step(x, u)
        discrepancies.append(np.abs(next_x-expected))
        forces.append(K_A*(u-next_x[0]))
        states.append(next_x.copy())
        x = next_x
    states = np.asarray(states)
    discrepancy = np.asarray(discrepancies).max(axis=0)
    if discrepancy[0] > PUSHER_ALLOWANCE or discrepancy[1] > OBJECT_ALLOWANCE:
        raise RuntimeError(f"Numerical allowance exceeded: {discrepancy}; investigate before interpreting bounds.")
    noise = rng.uniform(-POSITION_NOISE, POSITION_NOISE, len(states))
    noise[0] = initial_noise
    measurements = states[:,1] + noise
    fit = identify_positions(commands, measurements)
    mass_intervals = fit["mass_intervals"]
    covered = bool(np.all((mass_intervals[:,0]-1e-12 <= mass) & (mass <= mass_intervals[:,1]+1e-12)))
    force = np.asarray(forces)
    travel = states[:,1] - o0
    actual_a = np.asarray(commands) + W - states[:-1,1]
    limits_ok = bool(feasible(commands,Y0) and travel.max() <= MAX_TRAVEL+FEAS_TOL and
                     force.max() <= MAX_FORCE+K_A*FEAS_TOL and
                     actual_a.min() >= MIN_EXCITATION-FEAS_TOL)
    return dict(states=states, measurements=measurements, force=force, fit=fit,
                coverage=covered, limits_ok=limits_ok, discrepancy=discrepancy,
                travel_mm=1e3*travel[-1], width_g=1e3*np.diff(mass_intervals[-1])[0],
                peak_force_mN=1e3*force.max(),
                force_squared_integral=float(H*np.sum(force**2)),
                estimate_error_g=1e3*(fit["mass_estimate"]-mass))



DEMO_MASS, DEMO_SEED = .015, 7
DEMO, DEMO_FAILURES = {}, []
print(f"{'Strategy':30s} {'Travel mm':>10s} {'Peak mN':>9s} {'Width g':>9s} {'Abs err g':>10s} {'Covered':>8s} {'Limits':>7s}")
for name,u in PLANS.items():
    try:
        result = execute_and_evaluate(u,DEMO_MASS,DEMO_SEED)
    except Exception as exc:
        DEMO_FAILURES.append(dict(strategy=name,error=f"{type(exc).__name__}: {exc}"))
        print(name,"FAILED:",DEMO_FAILURES[-1]["error"])
        continue
    DEMO[name]=result
    print(f"{name:30s} {result['travel_mm']:10.4f} {result['peak_force_mN']:9.3f} "
          f"{result['width_g']:9.4f} {abs(result['estimate_error_g']):10.4f} "
          f"{str(result['coverage']):>8s} {str(result['limits_ok']):>7s}")
print("Demo runtime failures:",len(DEMO_FAILURES))
print("Demo coverage failures:",sum(not r['coverage'] for r in DEMO.values()))
print("Demo limit violations:",sum(not r['limits_ok'] for r in DEMO.values()))


## 16. Plot physical motion, measured learning, and the predicted score


In [ ]:
times=np.arange(N+1)*H
fig, axes=plt.subplots(1,3,figsize=(17,4.5))
colors={}
for name,r in DEMO.items():
    line,=axes[0].plot(times,1e3*(r["states"][:,1]-r["states"][0,1]),marker=".",label=name)
    colors[name]=line.get_color()
    mi=r["fit"]["mass_intervals"]*1e3
    axes[1].plot(np.arange(N+1),mi[:,1]-mi[:,0],marker=".",color=colors[name])
    axes[2].scatter(r["travel_mm"],r["width_g"],color=colors[name],label=name)
axes[0].set(xlabel="Time [s]",ylabel="Object displacement [mm]",title="Drake execution")
axes[1].set(xlabel="Observed transition",ylabel="Mass interval width [g]",title="Position-only identification")
axes[2].set(xlabel="Final displacement [mm]",ylabel="Final interval width [g]",title="Observed tradeoff")
for ax in axes:ax.grid(alpha=.3)
if DEMO:axes[0].legend(fontsize=6,ncol=2)
plt.tight_layout();plt.show()

fig,axes=plt.subplots(1,3,figsize=(17,4))
for name,r in DEMO.items():
    u=PLANS[name]; color=colors[name]
    axes[0].stairs((u-(Y0-W))*1e3,times,color=color,label=name)
    axes[1].stairs(r["force"]*1e3,times,color=color)
    forecast=forecast_with_gradient(u,Y0)
    axes[2].plot(np.arange(N+1),forecast["traces"],marker=".",color=color)
axes[0].set(xlabel="Time [s]",ylabel="Command from initial contact boundary [mm]")
axes[1].set(xlabel="Time [s]",ylabel="Model contact force [mN]")
axes[1].axhline(MAX_FORCE*1e3,color="black",linestyle="--",label="Force limit")
axes[2].set(xlabel="Forecast transition",ylabel="Normalized trace q",title="Prediction, not observed mass width",yscale="log")
for ax in axes:ax.grid(alpha=.3)
axes[1].legend(fontsize=7)
plt.tight_layout();plt.show()

fig,ax=plt.subplots(figsize=(6,4))
for name,r in DEMO.items():
    qN=forecast_with_gradient(PLANS[name],Y0)["traces"][-1]
    ax.scatter(qN,r["width_g"],color=colors[name],label=name)
ax.set(xlabel="Predicted terminal trace",ylabel="Observed mass width [g]",xscale="log",
       title="Inspect surrogate alignment; axes have different meanings")
ax.grid(alpha=.3);plt.tight_layout();plt.show()


## 17. Paired batch evaluation and direct comparison with the frozen method

Use the same fixed plans for every hidden mass and noise seed. Report failures in addition
to statistics for completed trials. Compare the **same lambda** across the two methods;
the trace objective and measured interval can favor different plans. No result is assumed
to improve. Selecting a lambda based on these outcomes would require fresh evaluation
data before claiming an independent test.


In [ ]:
ROWS=[]
if RUN_BATCH:
    for mass in EVALUATION_MASSES:
        print("Evaluating mass [g]:",mass*1e3,flush=True)
        for seed in EVALUATION_SEEDS:
            for name,u in PLANS.items():
                row=dict(strategy=name,mass_g=mass*1e3,seed=seed,completed=False,
                         width_g=np.nan,travel_mm=np.nan,peak_force_mN=np.nan,
                         estimate_error_g=np.nan,coverage=False,limits_ok=False,
                         max_object_error_um=np.nan,error="")
                try:
                    r=execute_and_evaluate(u,mass,seed)
                    row.update(completed=True,width_g=r["width_g"],travel_mm=r["travel_mm"],
                               peak_force_mN=r["peak_force_mN"],estimate_error_g=r["estimate_error_g"],
                               coverage=r["coverage"],limits_ok=r["limits_ok"],
                               max_object_error_um=r["discrepancy"][1]*1e6)
                except Exception as exc:
                    row["error"]=f"{type(exc).__name__}: {exc}"
                ROWS.append(row)
    print(f"{'Strategy':30s} {'Done/total':>11s} {'Mean width g':>13s} {'Mean move mm':>13s} {'MAE g':>10s} {'Cover(done)':>12s} {'Limit fails':>12s}")
    for name in PLANS:
        all_rows=[r for r in ROWS if r["strategy"]==name]
        done=[r for r in all_rows if r["completed"]]
        if not done:
            print(name,"NO COMPLETED TRIALS");continue
        print(f"{name:30s} {len(done):4d}/{len(all_rows):<6d} "
              f"{np.mean([r['width_g'] for r in done]):13.4f} "
              f"{np.mean([r['travel_mm'] for r in done]):13.4f} "
              f"{np.mean([abs(r['estimate_error_g']) for r in done]):10.4f} "
              f"{np.mean([r['coverage'] for r in done]):12.1%} "
              f"{sum(not r['limits_ok'] for r in done):12d}")
    print("Runtime failures:",sum(not r["completed"] for r in ROWS))
    for row in [r for r in ROWS if not r["completed"]][:5]:print("Failure:",row)
    index={(r["strategy"],r["mass_g"],r["seed"]):r for r in ROWS}
    print("Full-history vs frozen: same-weight paired comparisons")
    for weight in INFO_WEIGHTS:
        new,old_name=f"Full history lambda={weight:g}",f"Frozen lambda={weight:g}"
        if new not in PLANS or old_name not in PLANS:continue
        pairs=[]
        for mass in EVALUATION_MASSES:
            for seed in EVALUATION_SEEDS:
                a=index[(new,mass*1e3,seed)];b=index[(old_name,mass*1e3,seed)]
                if a["completed"] and b["completed"]:pairs.append((a,b))
        comparable=[(a,b) for a,b in pairs if a["coverage"] and b["coverage"] and a["limits_ok"] and b["limits_ok"]]
        wins=sum(a["width_g"]<=b["width_g"]+1e-9 and a["travel_mm"]<b["travel_mm"]-1e-6 for a,b in comparable)
        print(f"lambda={weight:g}: covered/feasible pairs={len(comparable)}/{len(pairs)}, "
              f"no-wider interval and lower displacement={wins}/{len(comparable)}")
        print("  full-objective new / frozen:",full_objective(PLANS[new],Y0,weight),
              full_objective(PLANS[old_name],Y0,weight))
    print("Coverage is empirical under the stated assumptions; no global-optimality claim follows.")
else:
    print("Batch disabled by RUN_BATCH=False.")


## 18. Optional export of plans, diagnostics, histories, and all evaluated outcomes


In [ ]:
def json_ready(value):
    if isinstance(value,np.ndarray):return json_ready(value.tolist())
    if isinstance(value,np.generic):return json_ready(value.item())
    if isinstance(value,dict):return {str(k):json_ready(v) for k,v in value.items()}
    if isinstance(value,(list,tuple)):return [json_ready(v) for v in value]
    if isinstance(value,float) and not np.isfinite(value):return None
    return value

if SAVE_RESULTS:
    folder=PROJECT/"results/information_aware_probing_full_history"/datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S_%fZ")
    folder.mkdir(parents=True,exist_ok=False)
    if ROWS:
        with (folder/"batch_results.csv").open("w",newline="") as stream:
            writer=csv.DictWriter(stream,fieldnames=list(ROWS[0]));writer.writeheader();writer.writerows(ROWS)
    np.savez_compressed(folder/"plans.npz",**PLANS)
    records=dict(plan_metadata=PLAN_META,optimization=OPT_RESULTS,
                 planning_failures=PLAN_FAILURES,demo_failures=DEMO_FAILURES,
                 diagnostics=DIAGNOSTICS,demo=DEMO)
    (folder/"run_details.json").write_text(json.dumps(json_ready(records),indent=2,allow_nan=False))
    settings={name:globals()[name] for name in [
        "K_A","H","W","N","MASS_PRIOR","POSITION_NOISE","OBJECT_ALLOWANCE","PUSHER_ALLOWANCE",
        "MIN_EXCITATION","MAX_TRAVEL","MAX_FORCE","MAX_COMMAND_STEP","MAX_COMMAND_OFFSET",
        "MOVE_SCALE","FORCE_SCALE","COMMAND_SCALE","MOVE_WEIGHT","FORCE_WEIGHT","COMMAND_WEIGHT",
        "INFO_WEIGHTS","OUTER_ITERS","FULL_MAX_ITERS","TRUST_INITIAL","TRUST_MIN","TRUST_MAX",
        "TRUST_SHRINK","TRUST_EXPAND","ACCEPT_RATIO","EXPAND_RATIO","PROX_REG","GAIN_REG",
        "FEAS_TOL","TARGET_TOL","STATIONARITY_TOL","PRED_REDUCTION_TOL","ACTUAL_REDUCTION_TOL",
        "RESIDUAL_BOUND","SENSITIVITY_THRESHOLD","FD_COMMAND_STEP","Y0","RUN_TARGET_MODE","TRACE_TARGET",
        "RUN_DIAGNOSTICS","RUN_FROZEN_COMPARISON","RUN_BATCH","EVALUATION_MASSES","EVALUATION_SEEDS",
        "DEMO_MASS","DEMO_SEED"]}
    source_files=[ALGORITHM_DIR/"traj_opt_problem.py",ALGORITHM_DIR/"iterative_to.py",source_path]
    settings["source_sha256"]={str(path):hashlib.sha256(path.read_bytes()).hexdigest() for path in source_files}
    settings["python"]=sys.version
    settings["notebook_version"]="05-full-history-2026-09-30"
    (folder/"settings.json").write_text(json.dumps(json_ready(settings),indent=2,allow_nan=False))
    print("Saved:",folder)
else:
    print("Set SAVE_RESULTS=True in Cell 2 or here, then rerun this cell after reviewing results.")


## Interpreting your run

1. Source parity and compact-recursion parity test the implementation of the chosen algebra.
2. Gradient checks assess local derivatives away from gate changes.
3. Trust-region logs show whether the optimization accepted improvements or stalled.
4. Drake discrepancies assess the numerical allowances only on evaluated states and commands.
5. Actual interval width, mass-estimate error, coverage, force, and motion evaluate usefulness.

A better terminal trace is not automatically a narrower observed mass interval. A narrow
interval that excludes truth is a failure. A small first-order gap is branch-local, not a
proof of a globally optimal probing trajectory. No precomputed results or success claims
are included in this notebook.
